In [17]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit, cross_val_score, GroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from pathlib import Path
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor
from tqdm_joblib import tqdm_joblib
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform, randint, uniform

DATA_DIR = Path().cwd().parent / 'data'

COLS_TO_DROP = ['game_id']
CV = 5

trainData = pd.read_csv(DATA_DIR / 'features.csv')

X = trainData.drop(columns=['elo'].copy())
y = trainData['elo'].copy()
groups = X['game_id']

In [18]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
train_idx, test_idx = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

cv = GroupKFold(n_splits=CV)

In [19]:
dummy_model_mean = DummyRegressor(strategy='mean')
dummy_model_median = DummyRegressor(strategy='median')

In [20]:
dummy_mean = cross_val_score(dummy_model_mean, X_train, y_train, cv=cv,
                                n_jobs=-1, groups=X_train['game_id'], scoring='neg_mean_absolute_error').mean() 

dummy_median = cross_val_score(dummy_model_median, X_train, y_train, cv=cv,
                                n_jobs=-1, groups=X_train['game_id'], scoring='neg_mean_absolute_error').mean()




print("-----Results-----")
print("Dummy mean: ", -dummy_mean)
print("Dummy median: ", -dummy_median)

-----Results-----
Dummy mean:  235.28248098284158
Dummy median:  235.24455230464545


In [21]:
def build_random_forest(**kwargs):
    main_pipeline = Pipeline([
        ('drop', ColumnTransformer([
            ('drop', 'drop', COLS_TO_DROP)
        ], remainder='passthrough')),
        ('regressor', RandomForestRegressor(**kwargs, n_jobs=1))
    ])
    return main_pipeline

In [ ]:
param_grid = {
    "regressor__n_estimators": [300],
    "regressor__max_depth": [15, 25],
    "regressor__min_samples_leaf": [5, 20, 50],
    "regressor__max_features": ["sqrt", 0.5, 1.0]
}

n_iter = 18

search = GridSearchCV(build_random_forest(), param_grid, cv=cv,
                      scoring='neg_mean_absolute_error', n_jobs=14)

with tqdm_joblib(total=n_iter*5, desc="Ajustando hiperparametros"):
    search.fit(X_train, y_train, groups=X_train['game_id'])

print("Mejores parámetros:", search.best_params_)
print("Mejor MAE CV:", -search.best_score_)

"""
Mejores parámetros: {'regressor__max_depth': 25, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 5, 'regressor__n_estimators': 300}
Mejor MAE CV: 195.92468546794163
"""

Ajustando hiperparametros:   0%|          | 0/90 [00:00<?, ?it/s]

Ajustando hiperparametros:   2%|▏         | 2/90 [06:32<3:58:23, 162.54s/it]/home/javier/Universidad/grupoAries/pruebasML/elo_predictor/venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Ajustando hiperparametros: 100%|██████████| 90/90 [1:56:26<00:00, 77.63s/it]   

Mejores parámetros: {'regressor__max_depth': 25, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 5, 'regressor__n_estimators': 300}
Mejor MAE CV: 195.92468546794163


"\nMejores parámetros: {'regressor__max_depth': 25, 'regressor__max_features': 'sqrt', 'regressor__min_samples_leaf': 5, 'regressor__n_estimators': 300}\nMejor MAE CV: 195.95724643056724\n"

In [23]:
def build_linear_regression():
    main_pipeline = Pipeline([
        ('drop', ColumnTransformer([('drop', 'drop', COLS_TO_DROP)], remainder='passthrough')),
        ('regressor', LinearRegression())
    ])
    return main_pipeline

In [ ]:
lr_result = cross_val_score(build_linear_regression(), X_train, y_train, cv=cv,
                                 scoring='neg_mean_absolute_error', n_jobs = -1, groups=X_train['game_id']).mean()

print("Linear Regressor (SVD): ", -lr_result)

"""
Linear Regressor (SVD): 209.8862
"""

Linear Regressor (SVD):  209.88622028184585


In [25]:
def build_sgd_regression(**kwargs):
    main_pipeline = Pipeline([
        ('drop', ColumnTransformer([('drop', 'drop', COLS_TO_DROP)], remainder='passthrough')),
        ('scaler', StandardScaler()),
        ('regressor', TransformedTargetRegressor(regressor= SGDRegressor(**kwargs, random_state=0),
                                   transformer=StandardScaler()
                                   ))
    ])
    return main_pipeline


In [ ]:
param_grid = {
    "regressor__regressor__penalty": ["l2", "l1", "elasticnet"],
    "regressor__regressor__alpha": [1e-6, 1e-5],
    "regressor__regressor__l1_ratio": [0.15, 0.5, 0.85],
    "regressor__regressor__eta0": [1e-4, 1e-3, 1e-2],
    "regressor__regressor__loss": ["squared_error", "huber", "epsilon_insensitive"],
}


n_iter= 162

search = GridSearchCV(build_sgd_regression(), param_grid,
                            cv=cv, scoring='neg_mean_absolute_error',
                            n_jobs=-1)

with tqdm_joblib(total=n_iter * CV, desc="Ajustando hiperparametros"):
        search.fit(X_train, y_train, groups=X_train['game_id'])

print("Mejores parámetros:", search.best_params_)
print("Mejor MAE CV:", -search.best_score_)

"""
Mejores parámetros: {'regressor__regressor__alpha': 1e-06, 'regressor__regressor__eta0': 0.0001, 'regressor__regressor__l1_ratio': 0.5, 'regressor__regressor__loss': 'epsilon_insensitive', 'regressor__regressor__penalty': 'elasticnet'}
Mejor MAE CV: 209.52618208195526
"""

Ajustando hiperparametros: 100%|██████████| 810/810 [04:33<00:00,  2.97it/s]

Mejores parámetros: {'regressor__regressor__alpha': 1e-06, 'regressor__regressor__eta0': 0.0001, 'regressor__regressor__l1_ratio': 0.5, 'regressor__regressor__loss': 'epsilon_insensitive', 'regressor__regressor__penalty': 'elasticnet'}
Mejor MAE CV: 209.52618208195526


"\nMejores parámetros: {'regressor__regressor__alpha': 1e-06, 'regressor__regressor__eta0': 0.0001, 'regressor__regressor__l1_ratio': 0.15, 'regressor__regressor__loss': 'epsilon_insensitive', 'regressor__regressor__penalty': 'elasticnet'}\nMejor MAE CV: 216.36424211275357\n"

In [27]:
def build_hgb(**kwargs):
    return Pipeline([
         ('drop', ColumnTransformer([('drop', 'drop', COLS_TO_DROP)], remainder='passthrough')),
        ('regressor', HistGradientBoostingRegressor(loss='absolute_error', max_iter=500,
                                                    early_stopping=True, **kwargs))
    ])

In [28]:
param_dist = {
    "regressor__learning_rate": loguniform(0.01, 0.2),
    "regressor__max_leaf_nodes": randint(15, 255),
    "regressor__max_depth": [None, 6, 10, 15],
    "regressor__min_samples_leaf": randint(20, 500),
    "regressor__l2_regularization": loguniform(1e-3, 10),
    "regressor__max_features": uniform(0.5, 0.5),   
}

n_iter = 40
search = RandomizedSearchCV(build_hgb(), param_dist, n_iter=n_iter, cv=cv,
                            scoring='neg_mean_absolute_error', n_jobs=-1, random_state=0)

with tqdm_joblib(total=n_iter * CV, desc="Ajustando hiperparametros"):
    search.fit(X_train, y_train, groups=X_train['game_id'])

print("Mejores parámetros:", search.best_params_)
print("Mejor MAE CV:", -search.best_score_)

"""
Mejores parámetros: {'regressor__l2_regularization': np.float64(0.999993080940865), 'regressor__learning_rate': np.float64(0.027642323786591195), 'regressor__max_depth': None, 'regressor__max_features': np.float64(0.5271689941696268), 'regressor__max_leaf_nodes': 247, 'regressor__min_samples_leaf': 275}
Mejor MAE CV: 193.38875945720784
"""

Ajustando hiperparametros: 219it [23:38,  6.48s/it]                         

Mejores parámetros: {'regressor__l2_regularization': np.float64(0.12229065947034364), 'regressor__learning_rate': np.float64(0.034632801362530156), 'regressor__max_depth': 15, 'regressor__max_features': np.float64(0.59316617166338), 'regressor__max_leaf_nodes': 129, 'regressor__min_samples_leaf': 203}
Mejor MAE CV: 193.35586339302938


"\nMejores parámetros: {'regressor__l2_regularization': np.float64(0.999993080940865), 'regressor__learning_rate': np.float64(0.027642323786591195), 'regressor__max_depth': None, 'regressor__max_features': np.float64(0.5271689941696268), 'regressor__max_leaf_nodes': 247, 'regressor__min_samples_leaf': 275}\nMejor MAE CV: 193.38875945720784\n"

In [30]:
model = search.best_estimator_

y_pred = model.predict(X_test)
print("MAE Test: ", mean_absolute_error(y_test, y_pred))

MAE Test:  193.37569038028116
